# DS 542 Spring 2026 Project 1

In this class, projects are basically longer (e.g. 2-week) homework assignments here you are expected to do more experimentation and coding.

* Your task for this project is to improve the training results for a fixed architecture model for the Tree-or-Not data set.
* Your model will be constrained to use the architecture in this template.
* To train a better model, you will instead explore picking appropriate early stopping rules, initialization, learning rate, regularization and data augmentation.


## Background

This notebook builds a model detecting trees in images.
The data set is available on GitHub at https://github.com/DL4DS/tree-or-not or on the Shared Compute Cluster at `/projectnb/dl4ds/materials/datasets/tree-or-not`.

The initial data set consists of about 500 pictures. Most of them are from the Boston area, but some are from around the globe. Most of them were taken outside, but some were taken inside or in more exotic locations. Many other factors such as lighting, weather, and confounding bushes will make this a challenging problem.

## Assignment Directions

1. Run the provided notebook and confirm basic functionality.
2. Inspect the dataset. Do you see any labeling errors? (5%)
2. Explore training improvements in each of the following categories and report the results - early stopping, initialization, learning rate, regularization and data augmentation. (25%)
3. Take the best configuration of each of the five training modifications and apply them all to see what overall improvement you can get. (30%)
4. Explain as best you can why each improvement was included or not included in the best performing model. (10%)
5. Save the best performing model for further evaluation by the auto-grader. (30%)


## Modules

In [ ]:
import imageio.v2 as imageio
import livelossplot
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torcheval.metrics

## GPU Access

In [ ]:
def to_gpu(t):
    if torch.cuda.is_available():
        return t.cuda()
    return t

def to_numpy(t):
    return t.detach().cpu().numpy()

device = to_gpu(torch.ones(1,1)).device
device

## Load Data

If you are not running on the SCC, fetch the data from https://github.com/dl4ds/tree-or-not as needed. For example, you could use `git clone` to make a local copy and update `data_dir` below.

If you are running on SCC, the data is already loaded so just leave `data_dir` as is.

In [ ]:
# If you are running locally, change this path to the local path of the tree-or-not data set.
data_dir = "/projectnb/dl4ds/materials/datasets/tree-or-not"


In [ ]:
# DO NOT CHANGE THIS CELL

image_width = 256

def load_data_set(data_set_name):
    labels = pd.read_csv(f"{data_dir}/{data_set_name}.tsv", sep="\t")

    file_names = []
    images = []
    targets = []
    for i in range(labels.shape[0]):
        row = labels.iloc[i]
        try:
            image = imageio.imread(f"{data_dir}/images{image_width}/{row['filename']}")[...]
        except:
            print("SKIPPING ", row['filename'], "MISSING")
            continue

        if image.shape[0] != image.shape[1] * 3 // 4:
            print("SKIPPING ", row['filename'], image.shape)
            continue

        # convert from 0-255 to 0.0-1.0
        image = image / 255
        # prepend axis with length one
        # image = image.reshape(1, *image.shape)
        image = torch.tensor(image, device=device, dtype=torch.float32)
        # permute image dimensions to put color channel first
        image = torch.permute(image, [2, 0, 1])

        file_names.append(row['filename'])
        images.append(image)
        targets.append(row["target"])

    images = torch.stack(images)

    targets = torch.tensor(targets, device=device, dtype=torch.float32)
    targets = targets.long()

    return (file_names, images, targets)

train_data_set = load_data_set("train")
for t in train_data_set[1:]:
    print("TRAIN", t.shape, t.dtype, t.device)
(train_file_names, train_X, train_Y) = train_data_set

validation_data_set = load_data_set("validation")
for t in validation_data_set[1:]:
    print("VALIDATION", t.shape, t.dtype, t.device)
(validation_file_names, validation_X, validation_Y) = validation_data_set

validation2_data_set = load_data_set("validation2")
for t in validation2_data_set[1:]:
    print("VALIDATION2", t.shape, t.dtype, t.device)
(validation2_file_names, validation2_X, validation2_Y) = validation2_data_set

## Inspect the Data Set

This is a handy way to inspect the dataset. Randomly choose 4 images at a time.

It is a common oversight to not look closely at the dataset. It's often worthwhile to ensure everything is labeled correctly.

### Inspect the training set

In [ ]:
# You can rerun this cell to randomly inspect different images.

idx = torch.randperm(train_X.shape[0])[:4]
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

label_names = {0: "Not Tree", 1: "Tree"}

for ax, i in zip(axes.flatten(), idx):
    ax.imshow(to_numpy(torch.permute(train_X[i, :, :, :], (1, 2, 0))))
    label = label_names.get(int(train_Y[i]), str(int(train_Y[i])))
    ax.set_title(f"{train_file_names[int(i)]}: {label}", fontsize=9)
    ax.axis("off")

plt.tight_layout()

### Inspect the validation set

In [ ]:
# You can rerun this cell to randomly inspect different images.

idx = torch.randperm(validation_X.shape[0])[:4]
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

label_names = {0: "Not Tree", 1: "Tree"}

for ax, i in zip(axes.flatten(), idx):
    ax.imshow(to_numpy(torch.permute(validation_X[i, :, :, :], (1, 2, 0))))
    label = label_names.get(int(validation_Y[i]), str(int(validation_Y[i])))
    ax.set_title(f"{validation_file_names[int(i)]}: {label}", fontsize=9)
    ax.axis("off")

plt.tight_layout()

### Inspect the 2nd Validation Set

In [ ]:
# You can rerun this cell to randomly inspect different images.

idx = torch.randperm(validation2_X.shape[0])[:4]
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

label_names = {0: "Not Tree", 1: "Tree"}

for ax, i in zip(axes.flatten(), idx):
    ax.imshow(to_numpy(torch.permute(validation2_X[i, :, :, :], (1, 2, 0))))
    label = label_names.get(int(validation2_Y[i]), str(int(validation2_Y[i])))
    ax.set_title(f"{validation2_file_names[int(i)]}: {label}", fontsize=9)
    ax.axis("off")

plt.tight_layout()

<!-- BEGIN QUESTION -->

## Dataset Evaluation (5%)

What is your assessment of the dataset? Are all the images labeled well? Do you see any anomolies? If so, list the image names and what you think the labels should be.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Model Building

> Do not modify the architecture of the network. You can add regularizations that apply during training but don't effect the architecture.

In [ ]:
class TreeNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_0 = torch.nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5, stride=2, device=device)
        self.conv_1 = torch.nn.Conv2d(in_channels=5, out_channels=5, kernel_size=5, stride=2, device=device)
        self.conv_2 = torch.nn.Conv2d(in_channels=5, out_channels=5, kernel_size=5, stride=2, device=device)
        self.conv_3 = torch.nn.Conv2d(in_channels=5, out_channels=5, kernel_size=5, stride=2, device=device)
        self.fc_3 = torch.nn.Linear(585, 2)

        self.relu = torch.nn.ReLU()

    def forward(self, X):

        X = self.conv_0(X)
        X = self.relu(X)

        X = self.conv_1(X)
        X = self.relu(X)

        X = self.conv_2(X)
        X = self.relu(X)

        X = self.conv_3(X)
        X = self.relu(X)

        # flatten channels and image dimensions
        X = X.reshape(X.shape[:-3] + (-1,))

        X = self.fc_3(X)

        return X

test_model = TreeNetwork().to(device)
test_output = test_model(train_X[:5,:,:,:])
assert test_output.shape == (5, 2)
del test_output

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()

In [ ]:
DEFAULT_EPOCHS = 500

def train_model(model_class, epochs=DEFAULT_EPOCHS, learning_rate=1e-4, **kwargs):
    """
    Train a model for a given number of epochs and plot the loss and accuracy for the training and validation sets.
    """
    model = model_class(**kwargs)
    if torch.cuda.is_available():
        model = model.cuda()

    model = torch.nn.DataParallel(model)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    liveloss = livelossplot.PlotLosses()
    for i in range(epochs):
        model.train()

        optimizer.zero_grad(set_to_none=True)
        prediction = model(train_X)
        loss = loss_function(prediction, train_Y)
        loss.backward()
        optimizer.step()

        if (i + 1) % 50 == 0:
            liveloss_updates = {}
            with torch.no_grad():
                model.eval()

                def get_metrics(metrics_prefix, metrics_X, metrics_Y):
                    metrics_prediction = model(metrics_X)

                    return {
                        f"{metrics_prefix}loss": loss_function(metrics_prediction, metrics_Y),
                        f"{metrics_prefix}accuracy": torcheval.metrics.functional.multiclass_accuracy(torch.argmax(metrics_prediction, dim=-1), metrics_Y)
                    }
                
                liveloss_updates.update(get_metrics("", train_X, train_Y))
                liveloss_updates.update(get_metrics("val_", validation_X, validation_Y))

            liveloss_updates = {k: to_numpy(v) for k, v in liveloss_updates.items()}
            liveloss.update(liveloss_updates,
                            current_step=i+1)
            liveloss.send()

    return model

# We use a fixed seed for reproducibility
torch.manual_seed(42)

test_model = train_model(TreeNetwork, epochs=1)
del test_model

In [ ]:
base_model = train_model(TreeNetwork, epochs=DEFAULT_EPOCHS)

## Training Improvements (25%)

In this section you will experiment with five different kinds of training improvements:
1. Early stoppiong
2. Initialization
3. Learning Rates
4. Regularization
5. Data Augmentation

**Your description must be specific to this data set and baseline training process.**

You do not need to describe how these methods work in general, and generalities may cost points for making your answer less concise.

You can train your models below with a different number of epochs than used above.

Warning:
Your training improvements will be sanity checked when models are built with them below.

If your training improvement does not improve the validation accuracy, then it will deemed inappropriate for this specific data set and architecture, and you will lose points here.

<!-- BEGIN QUESTION -->

### Early Stopping (5% of the 25%)

The baseline model overfits a lot.

Update the training strategy to implement early stopping to avoid entering the overfitting regime. There are multiple ways to do this.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

What are all the different ways you can implement early stopping?

Which way did you implement?

What was your validation accuracy and loss before and after early stopping?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Initialization (5% of the 25%)

Look up what the default weights and biases initialization method is used by PyTorch.

Implement the Kaimeng/He initialization method discussed in the lecture and book.

> Set the random seed to 42 to remove variance in your experiment.


In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

Did the initialization yield any training improvements? If so, why do you think that is?

What wasy your validation trianing and loss with and without your initialization?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Learning Rate (5% of the 25%)

Sometimes a particular learning rate, or simple fixed learning rate does not give the best results. Experiment with learning rates and learning rate schedules to see if you can get an improvement.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

Did learning rate help? If so, why do you think that is?

What's the best validation accuracy and loss you could achieve compared to the baseline?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Regularization (5% of the 25%)

Try different regularization methods to see which one or ones give the best results.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

What are different types of regularization methods?

Which ones did you try?

Which worked the best and why do you think that is?

What is the improvement validation accuracy and loss over baseline?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## Data Augmentation (5% of the 25%)

This is a relatively small dataset. In such cases data augmentation may help with generalization.

Try different data augmentations and report the results and improvements.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

What data augmentation methods did you use?

How much of an improvement did you get over the baseline validation accuracy and loss?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## Training All Combinations (30%)

Take the best configuration of each of the five training modifications and apply them all to see what overall improvement you can get.

Note that it is possible that some improvements negatively impact other improvements. If you want to be truly rigorous you would implement a training function that takes arguments for each of the improvements so you can sweep through all the combinations. In fact there are experiment tracking tools that help you do this like [Weights and Bias Sweeps](https://docs.wandb.ai/models/sweeps). It is free for students by the way.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

What is your validation accuracy and loss compared to the baseline?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## Validate Set 2 (10%)

Test your best model again with the validation2 data.

Use your existing models trained and do not retrain them using the validation2 data.

In [ ]:
# YOUR IMPLEMENTATION HERE
# You can use multiple cells if you wish.

...

Report you accuracy and loss of validation set 2 vs validation set 1.

Any surprises? If so, can you explain why?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

If your new validation results are poor, you may go back and refine your training improvements, but avoid using validation2 when you do so.

<!-- BEGIN QUESTION -->

## Save Best Model for Auto-Grader Evaluation (30%)

Use `torch.save` to save your best performing model as "best.pt".
Check the auto-grader results as soon as possible to confirm that it can load your model.

Hint: The auto-grader will check this model's accuracy on a withheld test set.
Review your results above and tweak your improvements as you feel appropriate.
But beware, overfitting on the visible data sets will likely lead to poor performance on the withheld test set.

In [ ]:
# Save your best model with torch.save as "best.pt"
# replace `base_model` with the name of your best model

# if model may be wrapped in DataParallel
to_save = base_model.module.state_dict() if hasattr(base_model, "module") else base_model.state_dict()
torch.save(to_save, "best.pt")


<!-- END QUESTION -->

